# KulimaIQ — Model Notebook

**Crop disease detection with location-aware context**

This notebook documents the KulimaIQ machine learning pipeline for smallholder farmers in East/Central Africa:

1. **Data visualization & data engineering** — training data distributions, sample images, crop grouping
2. **Model architecture** — MobileNetV2 transfer learning with custom classifier head
3. **Initial performance metrics** — accuracy, precision, recall, F1-score, confusion matrix
4. **Deployment (MVP)** — Flutter mobile app, FastAPI backend, Swagger UI / Postman
5. **Location-aware disease detection** — why climate and geography matter for symptoms and recommendations

> **Note:** The current checkpoint was trained on 20 disease classes (1,280 train / 320 val images). The app layers **farm GPS, weather, and climate advisories** on top of the vision model so recommendations can become location-specific as regional data grows.

## 1. Setup

In [ ]:
# Run from backend/ directory
# pip install torch torchvision matplotlib seaborn scikit-learn jupyter

import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from torch.utils.data import DataLoader
from torchvision import datasets

# Project imports
ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))
from app.ml.dataset import load_datasets, val_transform
from app.ml.model import build_model, load_model, INPUT_SIZE

DATA_DIR = ROOT / 'data'
WEIGHTS = ROOT / 'model_weights' / 'kulimaiq_mobilenet.pth'
HISTORY_PATH = ROOT / 'model_weights' / 'training_history.json'
METRICS_PATH = ROOT / 'notebooks' / 'eval_metrics.json'

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('PyTorch', torch.__version__)
print('Weights exist:', WEIGHTS.exists())

## 2. Data visualization & data engineering

### 2.1 Dataset layout

Training data follows an **ImageFolder** structure — each class is a folder named `{crop}_{disease}`:

```
data/
  train/
    healthy/
    cassava_mosaic/
    tomato_late_blight/
    ...
  val/
    (same class folders)
```

**Trusted public sources** used or supported by `scripts/prepare_data.py`:
- PlantVillage (HuggingFace)
- Kaggle datasets: Cassava, Bean, Rice, Banana, Coffee, Cotton, Mango, etc.

**Data engineering steps:**
| Step | Description |
|------|-------------|
| Download & map | Source dataset labels → KulimaIQ slug convention |
| Train/val split | 80/20 per class (or source-provided split) |
| Augmentation (train) | RandomResizedCrop, flips, ColorJitter, rotation |
| Normalization | ImageNet mean/std (MobileNetV2 pretrained) |
| Class weighting | Inverse frequency weights in CrossEntropyLoss |
| Label smoothing | 0.1 to reduce overconfidence |

In [ ]:
def count_images(split: str) -> dict[str, int]:
    base = DATA_DIR / split
    counts = {}
    for folder in sorted(base.iterdir()):
        if folder.is_dir():
            n = len(list(folder.glob('*.jpg')) + list(folder.glob('*.jpeg')) + list(folder.glob('*.png')))
            counts[folder.name] = n
    return counts

train_counts = count_images('train')
val_counts = count_images('val')
classes = sorted(train_counts.keys())

df = pd.DataFrame({
    'class': classes,
    'crop': [c.split('_')[0] for c in classes],
    'train': [train_counts[c] for c in classes],
    'val': [val_counts[c] for c in classes],
})
df['total'] = df['train'] + df['val']
display(df)

print(f"Total train: {df['train'].sum()} | Total val: {df['val'].sum()} | Classes: {len(classes)}")

In [ ]:
# Class distribution (train set)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(df['class'], df['train'], color='steelblue')
axes[0].set_title('Training images per class')
axes[0].set_xlabel('Count')

crop_train = df.groupby('crop')['train'].sum().sort_values(ascending=True)
axes[1].barh(crop_train.index, crop_train.values, color='darkorange')
axes[1].set_title('Training images grouped by crop')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Sample images from each crop (one class per crop)
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()
shown_crops = set()
idx = 0
for cls in classes:
    crop = cls.split('_')[0]
    if crop in shown_crops:
        continue
    shown_crops.add(crop)
    img_dir = DATA_DIR / 'train' / cls
    imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    if not imgs:
        continue
    img = Image.open(imgs[0]).convert('RGB')
    axes[idx].imshow(img)
    axes[idx].set_title(f"{crop}\n({cls})", fontsize=9)
    axes[idx].axis('off')
    idx += 1
    if idx >= len(axes):
        break
for j in range(idx, len(axes)):
    axes[j].axis('off')
plt.suptitle('Sample training images (one disease class per crop)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Train vs validation split ratio per class
df['val_ratio'] = df['val'] / df['total']
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(df['class'], df['val_ratio'], color='teal')
ax.axhline(0.2, color='red', linestyle='--', label='Target 20% val')
ax.set_xticklabels(df['class'], rotation=90, fontsize=7)
ax.set_ylabel('Validation fraction')
ax.set_title('Train/validation split balance')
ax.legend()
plt.tight_layout()
plt.show()

### 2.2 Correlation view (crop × disease presence)

Each class is a distinct label; the heatmap below shows which crops are represented in the training set (binary presence). As regional datasets are added, this matrix can expand with **location-specific variants** (e.g. `tomato_late_blight_ke` vs `tomato_late_blight_rw`).

In [ ]:
crop_disease = defaultdict(set)
for cls in classes:
    parts = cls.split('_', 1)
    crop = parts[0]
    disease = parts[1] if len(parts) > 1 else 'healthy'
    crop_disease[crop].add(disease)

all_diseases = sorted({d for ds in crop_disease.values() for d in ds})
matrix = pd.DataFrame(0, index=sorted(crop_disease.keys()), columns=all_diseases)
for crop, diseases in crop_disease.items():
    for d in diseases:
        matrix.loc[crop, d] = 1

plt.figure(figsize=(12, 6))
sns.heatmap(matrix, cmap='YlGnBu', cbar=False, linewidths=0.5)
plt.title('Crop × disease coverage in current training set')
plt.xlabel('Disease / condition slug')
plt.ylabel('Crop')
plt.tight_layout()
plt.show()

## 3. Model architecture

**Backbone:** MobileNetV2 (ImageNet pretrained) — efficient for mobile deployment.

**Custom classifier head:**
```
Dropout(0.3) → Linear(1280 → 512) → ReLU → Dropout(0.2) → Linear(512 → N classes)
```

**Optimization:**
| Component | Choice |
|-----------|--------|
| Loss | CrossEntropyLoss + class weights + label smoothing 0.1 |
| Optimizer | AdamW (lr=1e-3, weight_decay=1e-4) |
| Scheduler | CosineAnnealingLR (T_max = epochs) |
| Input size | 224 × 224 RGB |
| Activation (head) | ReLU |
| Output | Softmax over N disease classes |

In [ ]:
# Build architecture (same as production)
num_classes = len(classes)
model = build_model(num_classes=num_classes, pretrained=False)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Input size: {INPUT_SIZE}x{INPUT_SIZE}")

In [ ]:
# Layer summary table
rows = []
for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear, nn.BatchNorm2d, nn.Dropout)):
        params = sum(p.numel() for p in module.parameters())
        rows.append({'layer': name, 'type': module.__class__.__name__, 'params': params})
layer_df = pd.DataFrame(rows)
display(layer_df.tail(12))

## 4. Initial performance metrics

Metrics computed on the **validation set** using the saved checkpoint `model_weights/kulimaiq_mobilenet.pth`.

> **Caution:** Current training data includes synthetic/bootstrap images for pipeline testing. **Re-train on real PlantVillage + regional field images** before production deployment. Location-aware variants will further improve local accuracy.

In [ ]:
train_ds, val_ds = load_datasets(str(DATA_DIR))
model, class_names = load_model(str(WEIGHTS), device='cpu')
loader = DataLoader(val_ds, batch_size=32, shuffle=False)

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in loader:
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        y_true.extend(labels.tolist())
        y_pred.extend(preds.tolist())

acc = accuracy_score(y_true, y_pred)
prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)

metrics_df = pd.DataFrame([
    {'metric': 'Accuracy', 'value': acc},
    {'metric': 'Precision (weighted)', 'value': prec_w},
    {'metric': 'Recall (weighted)', 'value': rec_w},
    {'metric': 'F1-score (weighted)', 'value': f1_w},
    {'metric': 'Precision (macro)', 'value': prec_m},
    {'metric': 'Recall (macro)', 'value': rec_m},
    {'metric': 'F1-score (macro)', 'value': f1_m},
])
metrics_df['value'] = metrics_df['value'].round(4)
display(metrics_df)

print('\nPer-class report:')
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, xticklabels=class_names, yticklabels=class_names, cmap='Blues', fmt='d')
plt.title('Confusion matrix (validation set)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# Training curves from training_history.json
history = json.loads(HISTORY_PATH.read_text())
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train loss')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'], label='Val loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].set_title('Loss curves')

axes[1].plot(hist_df['epoch'], hist_df['train_acc'], label='Train acc')
axes[1].plot(hist_df['epoch'], hist_df['val_acc'], label='Val acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].set_title('Accuracy curves')
plt.tight_layout()
plt.show()

best = hist_df.loc[hist_df['val_acc'].idxmax()]
print(f"Best epoch: {int(best['epoch'])} | Val accuracy: {best['val_acc']:.4f}")

## 5. Location-aware disease detection

### Why location matters

The **same crop disease can look different** depending on:
- **Climate** — humidity, temperature, and rainfall change lesion colour, size, and spread
- **Altitude & soil** — stress symptoms can mimic disease (e.g. nutrient deficiency vs viral mosaic)
- **Season & growth stage** — young vs mature leaves show different patterns
- **Regional pathogen strains** — e.g. cassava mosaic severity varies across East Africa

**Example:** Tomato late blight (*Phytophthora infestans*) may show water-soaked lesions in cool, wet highland zones but faster necrosis in warmer lowland areas. Farmers in **Kenya vs Rwanda** may need different scouting cues and spray timing even when the label is the same.

### KulimaIQ MVP architecture (vision + location context)

```
┌─────────────────────┐     ┌──────────────────────────┐
│  Leaf photo (CNN)   │     │  Farm GPS + country/region│
│  MobileNetV2        │     │  Open-Meteo weather       │
└─────────┬───────────┘     └────────────┬─────────────┘
          │                              │
          └──────────┬───────────────────┘
                     ▼
          ┌──────────────────────┐
          │  FastAPI /analyze     │
          │  + recommendations    │
          │  + MongoDB history    │
          └──────────┬───────────┘
                     ▼
          ┌──────────────────────┐
          │  Flutter mobile app   │
          │  Scan · Farms · Weather│
          └──────────────────────┘
```

| Layer | Current MVP | Planned enhancement |
|-------|-------------|-------------------|
| Vision model | 20-class MobileNetV2 on leaf image | Fine-tune on regional field photos |
| Crop hint | User selects crop before scan | Filter softmax to crop-relevant classes |
| Farm location | GPS on map picker, stored in MongoDB | Agro-ecological zone tagging |
| Weather | Open-Meteo 7-day forecast per farm | Risk rules (humidity + disease) |
| Recommendations | `recommendations.json` by disease ID | Region-specific action text |
| Future ML | — | Multi-input model: image + climate embedding + geo region |

### Location integration in the API

When a farmer scans a leaf **linked to a farm**, the backend stores:
- `farm_id` → latitude, longitude, country, region
- `crop` selected by user
- `disease` + `confidence` from CNN
- `recommendation`, `severity`, `actions`

The Flutter app also shows **live weather advisories** per farm location (rain, heat, dry spell) to contextualize disease risk — because environmental conditions affect both symptom expression and treatment timing.

In [ ]:
# Example: crop-filtered inference concept (post-processing)
# After CNN predicts all classes, boost or filter by selected crop

def filter_probs_by_crop(all_probs: dict[str, float], crop: str) -> dict[str, float]:
    """Keep only classes belonging to the selected crop (+ healthy)."""
    filtered = {k: v for k, v in all_probs.items()
                if k == 'healthy' or k.startswith(f'{crop}_')}
    if not filtered:
        return all_probs
    total = sum(filtered.values())
    return {k: v / total for k, v in filtered.items()}

example_probs = {'tomato_late_blight': 0.72, 'cassava_mosaic': 0.15, 'healthy': 0.13}
print('Raw:', example_probs)
print('Filtered for crop=tomato:', filter_probs_by_crop(example_probs, 'tomato'))

## 6. Deployment — MVP / Mockup

KulimaIQ is deployed as a **three-tier MVP**:

| Tier | Technology | Role |
|------|------------|------|
| Mobile | Flutter (`kulimaiq_app/`) | Scan, farms, weather, profile, offline cache |
| API | FastAPI + Uvicorn (`backend/`) | Auth, ML inference, farms, diagnoses |
| Database | MongoDB | Users, farms, scan history |
| ML | PyTorch MobileNetV2 | Disease classification |

### 6.1 API (Swagger UI & Postman)

Start the backend:
```bash
cd backend
source .venv/bin/activate
uvicorn app.main:app --reload --port 8001
```

| Resource | URL |
|----------|-----|
| Swagger UI | http://localhost:8001/docs |
| ReDoc | http://localhost:8001/redoc |
| Health | http://localhost:8001/health |

**Key endpoints for demo:**

| Method | Endpoint | Purpose |
|--------|----------|---------|
| POST | `/auth/register` | Create farmer account |
| POST | `/auth/login` | JWT token |
| POST | `/analyze/image` | Upload leaf photo (multipart) |
| GET | `/diagnoses/` | Scan history |
| GET/POST | `/farms/` | Farm CRUD with GPS |
| GET | `/crops/classes` | Model class list |

**Postman flow:**
1. Register → copy `access_token`
2. Set header: `Authorization: Bearer <token>`
3. POST `/analyze/image` with form fields `crop`, optional `farm_id`, file `image`
4. GET `/diagnoses/` to verify MongoDB persistence

### 6.2 Mobile app (Flutter)

```bash
cd kulimaiq_app
flutter run
```

**Screens:** Home (recent scans) · Scan (camera + crop grid) · Weather (Open-Meteo advisories per farm GPS) · Farms (map picker, health score) · Profile (backend URL, language)

Default backend URL (Android emulator): `http://10.0.2.2:8001`

**Demo account:** `+250788000000` / `farmer123`

### 6.3 Web interface

FastAPI serves interactive **Swagger UI** at `/docs` — sufficient for MVP API testing. A dedicated farmer web portal can be added later (React/Flutter Web) using the same REST API.

In [ ]:
# Optional: verify API health (requires server running on port 8001)
import urllib.request

try:
    with urllib.request.urlopen('http://localhost:8001/health', timeout=3) as resp:
        health = json.loads(resp.read().decode())
    print('API status:', health)
except Exception as e:
    print('API not reachable (start with: uvicorn app.main:app --port 8001)')
    print('Error:', e)

## 7. Summary

| Item | Result |
|------|--------|
| Model | MobileNetV2 + custom 512-unit head |
| Classes | 20 (multiple crops + healthy) |
| Val accuracy | ~99.7% (bootstrap/synthetic data — re-train on real images) |
| Location | Farm GPS + weather advisories + future regional model variants |
| Deployment | Flutter mobile + FastAPI + MongoDB + Swagger |